# 02 — Feature families (proposal 3.2)

In [1]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # so `import src...` works from notebooks/

import numpy as np
import pandas as pd

from src import config
from src.utils import set_seed, save_fig
set_seed()  # fix all RNGs -- reproducibility

Build in cost order: **stylometric → sbert → biber**. Each one: build, then sanity-check it alone with `train_and_evaluate`. Cache SBERT + POS.

In [2]:
from src import data, features, modeling
clean = data.load_or_build_clean()
splits = data.load_or_build_splits(clean)
y = clean[config.LABEL_COL].values
texts = clean['text']

### Stylometric (write first)

In [3]:
sty = features.build_stylometric(texts)
sty_scaled, _ = features.scale_dense(sty.values, splits['train'])
sty.describe()

,avg_sentence_len,avg_word_len,ttr,mtld,comma_ratio,period_ratio,question_ratio,exclam_ratio
count,159288.000000,159288.000000,159288.000000,159288.000000,159288.000000,159288.000000,159288.000000,159288.000000
mean,32.071211,4.663111,0.462342,59.387071,0.010087,0.009449,0.000235,0.000143
std,53.120113,0.655728,0.202527,31.118980,0.008741,0.005529,0.004218,0.001011
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,15.076923,4.269949,0.390244,37.106634,0.005208,0.006823,0.000000,0.000000
50%,19.529412,4.662291,0.519608,58.986133,0.008895,0.008571,0.000000,0.000000
75%,24.000000,5.023649,0.595745,79.058511,0.012441,0.011470,0.000000,0.000000
max,1441.000000,42.103448,1.000000,618.520000,0.342812,0.324898,0.998532,0.242392


### SBERT (cache!)

In [4]:
emb = features.build_sbert(texts)   # cached to artifacts/sbert_embeddings.pkl
emb.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2489 [00:00<?, ?it/s]

(159288, 384)

### Biber (slow — cache POS tags)

In [7]:
bib = features.build_biber(texts)
bib_scaled, _ = features.scale_dense(bib.values, splits['train'])
bib.head()

,pos_noun,pos_propn,pos_verb,pos_aux,pos_adj,pos_adv,pos_pron,pos_det,pos_adp,pos_cconj,pos_sconj,pos_part,pos_intj,pos_num,modal_ratio,passive_ratio,nominalisation_ratio,discourse_marker_ratio
0,0.208633,0.068345,0.093525,0.017986,0.107914,0.017986,0.010791,0.057554,0.107914,0.046763,0.003597,0.010791,0.0,0.003597,0.000000,0.021583,0.032374,0.000000
1,0.298913,0.010870,0.076087,0.027174,0.108696,0.027174,0.038043,0.070652,0.097826,0.027174,0.000000,0.010870,0.0,0.000000,0.000000,0.010870,0.076087,0.014599
2,0.243542,0.000000,0.114391,0.029520,0.147601,0.025830,0.036900,0.077491,0.110701,0.014760,0.007380,0.022140,0.0,0.007380,0.007380,0.007380,0.044280,0.000000
3,0.297935,0.014749,0.103245,0.029499,0.094395,0.014749,0.029499,0.082596,0.106195,0.023599,0.008850,0.008850,0.0,0.005900,0.008850,0.023599,0.056047,0.000000
4,0.279152,0.017668,0.081272,0.024735,0.116608,0.028269,0.042403,0.109541,0.084806,0.031802,0.010601,0.010601,0.0,0.003534,0.014134,0.014134,0.074205,0.004310


### Sanity: each block alone

In [8]:
blocks = {"stylometric": sty_scaled, "sbert": emb, "biber": bib_scaled}
ytr, yval = y[splits['train']], y[splits['val']]

for name, X in blocks.items():
    res = modeling.train_and_evaluate('logreg', X[splits['train']], ytr, X[splits['val']], yval)
    print(f"{name}: Macro-F1 = {res.macro_f1:.4f}")

stylometric: Macro-F1 = 0.2894
sbert: Macro-F1 = 0.3728
biber: Macro-F1 = 0.2317


### Reading the sanity numbers

Random-guess floor for 12 balanced classes is Macro-F1 ≈ 0.08. All three blocks
(stylometric 0.289, sbert 0.373, biber 0.232) sit well above that, so each one
carries real attribution signal — the sanity check's job was just to confirm
the pipeline isn't leaking/broken, not to be the final read. Three things worth
carrying into `03_ablation` and the report:

1. **Ranking (sbert > stylometric > biber) is not an apples-to-apples
   comparison.** Dimensionality differs by orders of magnitude (tfidf 50k,
   sbert 384, biber 18, stylometric 8), and a linear classifier's capacity
   scales with feature count. The real test is Exp1 vs Exp2 vs Exp3 as
   standalone conditions in `03_ablation`, plus the combined conditions
   (Exp4/5/6) to see whether style features add anything on top of lexical.
2. **All three are far below TF-IDF-only (0.742 from `01_baseline_minimal`),
   which cuts against the proposal's stated hypothesis** ("style-based
   representations will prove more discriminative than purely lexical
   approaches"). Worth flagging explicitly in the report/discussion: the 11
   LLMs may be quite similar in grammar/style (shared RLHF-style tuning) but
   more separable in raw word choice — which also means TF-IDF's edge could
   partly be prompt/topic leakage rather than "true" source style, per the
   proposal's own caveat that lexical features "may partly reflect prompt
   wording or topic overlap."
3. **Biber (18 dims) scoring below stylometric (8 dims) is a bit odd** given
   it has more features. Possible explanation: Biber's feature set was
   designed to separate *register/genre*, not *author* — within one
   domain/prompt, the 12 sources may write in a fairly similar register, so
   POS-distribution and grammatical-marker features may carry less
   source-specific signal than expected. Also worth noting our
   `nominalisation_ratio`/`discourse_marker_ratio` are cheap heuristics
   (suffix list / fixed keyword list), not a validated NLP method — part of
   the weak biber signal could just be measurement noise in those two
   columns specifically.

This sanity SBERT-alone number (0.373) is effectively proposal Exp 3 and can
be reused directly in `03_ablation`'s results table.